# Sahayak — Fine-tune Gemma 4 **E4B** (QLoRA) on Kaggle **GPU T4 ×2**

End-to-end: install → GPU guard → HF auth → dataset validate (hard gate) → **E4B QLoRA** train →
merged-fp16 + GGUF export → upload to a private HF repo. Companion to `docs/SAHAYAK_DATASET_SPEC.md`
and `finetune/`.

### Before you run
1. **Settings → Accelerator = `GPU T4 x2`** (never P100 — it's compute-capability 6.0; Unsloth needs ≥ 7.0. T4 = 7.5). **Internet = On**.
2. **Add-ons → Secrets → add `HF_TOKEN`** = a Hugging Face token whose account has **accepted the Gemma license** at [huggingface.co/google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it). *Never paste the token in a cell.*
3. **Add Input → your dataset** containing `train.jsonl`, `val.jsonl`, and (optional) `eval_holdout.jsonl`. Cell 5 auto-finds them under `/kaggle/input`.

The script is single-GPU; the **second T4 idles** (harmless). Why E4B and not E2B: per Unsloth's Gemma 4 guide, E4B QLoRA beats E2B LoRA at similar memory — quantization costs less accuracy than the smaller model does, and T4's 16 GB fits E4B 4-bit (~10 GB).

## Cell 1 — install the fine-tune stack

Kaggle's image already ships torch + CUDA; we add **Unsloth**, which pulls a Gemma-4-compatible
`transformers`/`trl`/`peft`/`datasets`/`bitsandbytes`. We install Unsloth **directly** (not the repo's
`requirements.txt`, whose upper-bound pins predate Gemma 4, which needs `transformers ≥ 4.53`).

> If pip prints a torch/transformers conflict, do **Run → Restart & clear cell outputs**, then re-run from Cell 1.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# Unsloth (+ its resolver for a compatible transformers/trl/peft/datasets/bitsandbytes).
pip('-U', 'unsloth', 'unsloth_zoo')
# For the artifact upload step.
pip('-U', 'huggingface_hub[hf_transfer]')
print('\ninstall complete')

## Cell 2 — GPU guard (reject P100, confirm T4)

Fails loudly *before* a long run if the accelerator is wrong. Unsloth requires CUDA compute capability ≥ 7.0.

In [ ]:
import torch, subprocess

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), "No CUDA GPU visible. Settings -> Accelerator -> 'GPU T4 x2'."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}   compute capability: {major}.{minor}   VRAM: {vram:.1f} GB')
assert major >= 7, (
    f'Compute capability {major}.{minor} < 7.0 (e.g. Tesla P100 = 6.0). Unsloth needs >= 7.0. '
    "Switch the accelerator to 'GPU T4 x2' (T4 = 7.5)."
)
print('GPU OK for Unsloth QLoRA.')

## Cell 3 — Hugging Face auth (from Kaggle Secrets)

The token is read from the `HF_TOKEN` **secret** — it is never written into the notebook or any file.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    # Reduce allocator fragmentation on the 16 GB T4 (helps avoid load-time OOM spikes).
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    raise SystemExit(
        'Could not load HF_TOKEN from Kaggle Secrets. Add-ons -> Secrets -> add HF_TOKEN '
        '(a token whose HF account has accepted the Gemma license at '
        'huggingface.co/google/gemma-4-E4B-it). Original error: ' + repr(e)
    )

## Cell 4 — get the training code

Clones the repo for `sahayak_finetune.py` + `validate_dataset.py` and `cd`s into `finetune/` (so the
validator can import the canonical `SYSTEM_PROMPT` from the trainer). If your repo is private, either
make it public for the clone or upload the `finetune/` folder as a Kaggle dataset and point `FT` at it.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/SivaNithishKumar/sankat-mochan.git'
WORK = '/kaggle/working'
repo_dir = os.path.join(WORK, 'sankat-mochan')
if not os.path.isdir(repo_dir):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo_dir], check=True)

FT = os.path.join(repo_dir, 'finetune')
assert os.path.exists(os.path.join(FT, 'sahayak_finetune.py')), 'trainer script missing after clone'
assert os.path.exists(os.path.join(FT, 'validate_dataset.py')), 'validator missing after clone'
os.chdir(FT)
print('finetune dir:', FT)

## Cell 5 — locate the dataset & validate (HARD GATE)

Auto-finds the JSONL under `/kaggle/input`. The mechanical validator (`SAHAYAK_DATASET_SPEC.md` §8.1) is a
**hard gate** — training never runs on a file that fails it. `--eval` uses the **validation split**
(`val.jsonl`), *not* the held-out `eval_holdout.jsonl` (that is for the final hand-review only).

In [ ]:
import glob, os, subprocess, sys

def find_one(names):
    for n in names:
        hits = sorted(glob.glob(f'/kaggle/input/**/{n}', recursive=True))
        if hits:
            return hits[0]
    return None

TRAIN = find_one(['train.jsonl', 'all.jsonl'])
VAL   = find_one(['val.jsonl'])
HOLD  = find_one(['eval_holdout.jsonl', 'holdout.jsonl'])
print('train  :', TRAIN)
print('val    :', VAL)
print('holdout:', HOLD)
assert TRAIN and VAL, 'Attach a Kaggle dataset with train.jsonl and val.jsonl (Add Input).'

for f in [TRAIN, VAL]:
    rc = subprocess.run([sys.executable, 'validate_dataset.py', f]).returncode
    assert rc == 0, f'validation FAILED for {f} - fix the data before training.'
print('\ndataset validation passed.')

## Cell 6 — train E4B (QLoRA)

Best-accuracy settings: **r=32 / α=32**, **lr 2e-4**, **3 epochs**, **seq-len 1024**, effective batch **8**
(batch 1 × grad-accum 8 — the lowest-memory way to reach batch 8 on a 16 GB T4). The base model is the
**pre-quantized 4-bit** `unsloth/gemma-4-E4B-it-unsloth-bnb-4bit`: it loads straight as 4-bit (~6 GB) rather
than materializing fp16 weights first (which peaks ~17 GB and OOMs the T4 at load). Response masking
(`train_on_responses_only`) and the `gemma-4` chat template are applied automatically — watch the startup log
for the `[markers]` line and a falling **eval** loss each epoch.

_Full training; expect ~1–1.5 h including exports._

> **If you still hit an OOM during training** (not load): lower `--max-seq-len` to 768. Still OOM, or you want
> a guaranteed-fit run? Switch `--model` to **`unsloth/gemma-4-E2B-it-unsloth-bnb-4bit`** (the smaller E2B) —
> it trains on the T4 with wide headroom; everything else stays identical.

In [ ]:
import subprocess, sys

OUT = '/kaggle/working/sahayak-e4b'
cmd = [
    sys.executable, 'sahayak_finetune.py',
    '--train', TRAIN,
    '--eval',  VAL,
    # Pre-quantized 4-bit checkpoint: loads directly as 4-bit (~6 GB) instead of materializing
    # fp16 weights first, which peaks ~17 GB and OOMs a 16 GB T4 at load time.
    '--model', 'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit',
    '--out',   OUT,
    '--epochs', '3',
    '--lr', '2e-4',
    # batch 1 x grad-accum 8 = effective batch 8, but lowest activation memory on the T4.
    '--batch-size', '1',
    '--grad-accum', '8',
    '--lora-r', '32',
    '--lora-alpha', '32',
    '--max-seq-len', '1024',
    # SHARD the model's layers across BOTH T4s (Kaggle gives 2) = model parallelism, ~29 GB
    # combined VRAM. This is NOT DDP (which replicates the model per GPU and can't relieve OOM).
    # Delete this one line to fall back to a single T4 — the script forces bf16, which also fits
    # E4B on one card. Requires a single process (this is one) — never with torchrun/accelerate.
    '--device-map', 'balanced',
    '--export-merged',
    '--export-gguf', 'q4_k_m',
]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True)

> **On the GGUF step:** adapters (`OUT/`) and `merged-16bit/` are saved *before* GGUF conversion, so they
> are safe even if GGUF fails. GGUF for a brand-new architecture can need a current `llama.cpp`; if that step
> errors, use **`merged-16bit/`** (it's also the Qualcomm AI Hub input) and convert to GGUF later.

## Cell 7 — quick qualitative check on the holdout _(optional)_

Loads the fine-tuned adapters and generates a few held-out answers with Gemma 4's recommended sampling
(`temp 1.0, top_p 0.95, top_k 64`). Eyeball packet format, refusal calibration, length, and language.

In [ ]:
import json, torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from sahayak_finetune import SYSTEM_PROMPT

model, tok = FastModel.from_pretrained(OUT, max_seq_length=1024, load_in_4bit=True)
tok = get_chat_template(tok, chat_template='gemma-4')
try:
    FastModel.for_inference(model)
except Exception:
    pass

prompts = []
if HOLD:
    with open(HOLD, encoding='utf-8') as f:
        for line in f:
            o = json.loads(line)
            u = next((m['content'] for m in o['messages'] if m['role'] == 'user'), None)
            if u:
                prompts.append(u)
            if len(prompts) >= 3:
                break

for u in prompts:
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': u}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=True,
                         temperature=1.0, top_p=0.95, top_k=64)
    print('USER    :', u)
    print('SAHAYAK :', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print('-' * 80)

## Cell 8 — ship artifacts to a private HF repo

Uploads `adapters + merged-16bit + gguf` to `&lt;your-hf-user&gt;/sahayak-e4b` (private). The repo owner is
resolved from your token, so no username is hard-coded.

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
user = api.whoami()['name']
repo = f'{user}/sahayak-e4b'
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=repo)
print('uploaded ->', repo)
print('pull locally:  huggingface-cli download', repo, '--local-dir ./sahayak-e4b')

## Artifacts & next step (NPU)

You get three artifacts — keep all three:
- `gguf/*.gguf` (~4 GB, q4_k_m) → straight into the app's model import (llama.cpp CPU/GPU path).
- `merged-16bit/` (~16 GB safetensors) → the **Qualcomm AI Hub** input for NPU compilation.
- adapter files at the root (~100–200 MB) → re-merge later without retraining.

**Inference settings to bake into the app (must match at demo):** `temperature=1.0, top_p=0.95, top_k=64`,
the **`gemma-4`** chat template, and the **byte-identical** system prompt from the spec.

**NPU handoff (not on Kaggle):** AI Hub's LLM flow consumes the HF checkpoint (`merged-16bit/`), never GGUF —
quantize (AIMET, Linux/WSL + big-VRAM GPU) then export to QNN/Genie for `Snapdragon X Elite CRD`. See the
project's Kaggle guide §5 and `PLAN.md`.